# 03 · Physics-Informed Neural Network (Poisson-constrained)

### Physics
In the thin-lens approximation, with angular position $\theta$, source position $\beta$,
lensing potential $\psi$ and convergence (projected surface mass density) $\kappa$:

$$\alpha(\theta) = \nabla\psi(\theta),\qquad \beta = \theta - \alpha(\theta),\qquad \nabla^2\psi = 2\kappa .$$

### Model
An EfficientNet-B0 encoder (same as notebook 02) feeds a **physics decoder** that predicts
$\hat\psi$ and $\hat\kappa \ge 0$ on a 64×64 grid. From these:

1. **Deflection:** $\hat\alpha = \nabla\hat\psi$ (central finite differences).
2. **Lens equation:** the observed image is ray-traced back to the source plane,
   $\hat S(\theta) = I(\theta - \hat\alpha(\theta))$, via differentiable `grid_sample`.
3. A small **source encoder** reads $[\hat S, \hat\kappa]$; its features are concatenated with the
   image features for classification.

### Loss: the Poisson equation embedded in the objective
$$\mathcal L = \mathrm{CE}(\text{logits}, y) + \lambda\;\big\langle(\nabla^2\hat\psi - 2\hat\kappa)^2\big\rangle$$

with $\nabla^2$ the 5-point finite-difference Laplacian scaled by the grid spacing.

- The Poisson term forces the predicted **potential and mass density to be mutually consistent**.
- Since $\hat\kappa$ is constrained non-negative (softplus), the Poisson term also forces
  $\hat\psi$ to be **subharmonic**, i.e. generated by a non-negative mass distribution. A noisy
  or unphysical potential has regions of negative Laplacian and is penalised.
- Neither map can collapse to a trivial solution: CE supervises $\hat\psi$ through the
  ray-traced source and $\hat\kappa$ directly.
- **λ = 0** trains the identical architecture with the physics switched off, which is the ablation.

> The earlier version computed the Laplacian of the *input image* and regressed a sigmoid κ̂ onto
> it. That is an auxiliary image-filter target, not a Poisson constraint: it never
> differentiates a network output, and a sigmoid cannot match the negative values of ∇²I.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import torch

from deeplense import CLASS_NAMES
from deeplense.utils import get_device, seed_everything

seed_everything(42)
DEVICE = get_device()
DATA_ROOT = ROOT / "data" / "lensing"
RESULTS = ROOT / "results"
print("device:", DEVICE)

In [ ]:
# ── Run mode ──────────────────────────────────────────────────────────────────
# SMOKE = True : a few hundred images, 2 epochs, just to check everything runs (laptop).
# SMOKE = False: full dataset and full recipe (GPU recommended).
# If a full run already exists in results/<run>/ (e.g. from scripts/train.py),
# it is loaded instead of retraining unless RETRAIN = True.
SMOKE = True
RETRAIN = False
LAMBDA = 0.1          # physics-loss weight; see the ablation at the end

## Sanity check of the finite-difference physics

In [ ]:
from deeplense.physics import gradient, identity_grid, laplacian, poisson_residual, ray_trace

# Analytic test: psi = (x^2 + y^2) / 2  ->  laplacian = 2, grad = (x, y), so kappa = 1 solves Poisson exactly.
g = identity_grid(1, 64, 64)
psi = 0.5 * (g[..., 0] ** 2 + g[..., 1] ** 2)[None]
print("laplacian (should be 2):", laplacian(psi).mean().item())
print("Poisson residual with kappa=1:", poisson_residual(psi, torch.ones_like(psi)).abs().max().item())

# Point-mass-like lens (softened SIS): deflection toward the centre turns a compact source into a ring.
r = torch.sqrt(g[..., 0] ** 2 + g[..., 1] ** 2 + 1e-3)
psi_sis = (0.35 * r)[None]
src = torch.exp(-((g[..., 0] - 0.05) ** 2 + g[..., 1] ** 2) / 0.005)[None]
alpha = gradient(psi_sis)
lensed = ray_trace(src, alpha)
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, im, t in zip(axes, [src[0, 0], psi_sis[0, 0], lensed[0, 0]], ["source S(β)", "potential ψ (SIS)", "lensed I(θ) = S(θ − ∇ψ)"]):
    ax.imshow(im, cmap="inferno"); ax.set_title(t); ax.axis("off")
plt.show()

## Train (or load)

In [ ]:
from dataclasses import replace
from deeplense.data import DataConfig, build_loaders
from deeplense.metrics import predict
from deeplense.train import fit
from deeplense.utils import load_json
sys.path.insert(0, str(ROOT / "scripts"))
from train import RECIPES

cfg = RECIPES["pinn"]
cfg = replace(cfg, extra={"lambda_poisson": LAMBDA})
from deeplense.models import LensingPINN, PINNLoss
data_cfg = DataConfig(root=str(DATA_ROOT), num_workers=2)
if SMOKE:
    cfg = replace(cfg, epochs=2)
    data_cfg = replace(data_cfg, train_per_class=300, test_per_class=100)
RUN = f"pinn_lambda{LAMBDA:g}" + ("_smoke" if SMOKE else "")

loaders = build_loaders(data_cfg)
model = LensingPINN(pretrained=True)
criterion = PINNLoss(LAMBDA, cfg.label_smoothing)
run_dir = RESULTS / RUN

if (run_dir / "best.pt").exists() and not RETRAIN:
    model.load_state_dict(torch.load(run_dir / "best.pt", map_location="cpu"))
    model.to(DEVICE)
    results = load_json(run_dir / "metrics.json")
    history = load_json(run_dir / "history.json")
    probs, labels = predict(model, loaders["test"], DEVICE)
    print(f"Loaded {run_dir}  (best epoch {results['best_epoch']})")
else:
    model, out = fit(model, loaders, cfg, DEVICE, criterion=criterion, out_dir=RESULTS, run_name=RUN)
    results, history, probs, labels = out, out["history"], out["probs"], out["labels"]

## Evaluation on the held-out test set

In [ ]:
from deeplense.metrics import classification_metrics, plot_confusion, plot_history, plot_roc

plot_history(history, title=RUN); plt.show()

m = classification_metrics(probs, labels)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_roc(probs, labels, title=RUN, ax=axes[0])
plot_confusion(m["confusion_matrix"], title="Test confusion matrix", ax=axes[1])
plt.tight_layout(); plt.savefig(RESULTS / RUN / "roc_confusion.png", dpi=130, bbox_inches="tight"); plt.show()

print(f"Test accuracy {m['accuracy']:.4f} | macro AUC {m['macro_auc']:.4f}")
for k, v in m["auc_per_class"].items():
    print(f"  {k:<20} AUC {v:.4f}")

## What the physics branch learned

For one test image per class: the input, the predicted potential $\hat\psi$, the predicted
convergence $\hat\kappa$, the Poisson residual $\nabla^2\hat\psi - 2\hat\kappa$, and the ray-traced source $\hat S$.

In [ ]:
from deeplense.physics import poisson_residual

test_ds = loaders["test"].dataset
model.eval()
fig, axes = plt.subplots(3, 5, figsize=(19, 11))
for c in range(3):
    i = next(j for j, y in enumerate(test_ds.labels) if y == c)
    x = test_ds[i][0][None].to(DEVICE)
    with torch.no_grad():
        o = model(x)
    res = poisson_residual(o["psi"], o["kappa"])[0, 0].cpu()
    v = res.abs().max().item()
    panels = [(x[0, 0].cpu(), "input I", "inferno", None),
              (o["psi"][0, 0].cpu(), "potential ψ̂", "viridis", None),
              (o["kappa"][0, 0].cpu(), "convergence κ̂", "magma", None),
              (res, "∇²ψ̂ − 2κ̂", "RdBu_r", (-v, v)),
              (o["source"][0, 0].cpu(), "ray-traced source Ŝ", "inferno", None)]
    for ax, (im, t, cmap, lim) in zip(axes[c], panels):
        ax.imshow(im, cmap=cmap, **({"vmin": lim[0], "vmax": lim[1]} if lim else {}))
        ax.set_title(t if c == 0 else ""); ax.axis("off")
    axes[c, 0].text(-10, 75, CLASS_NAMES[c], rotation=90, va="center", ha="right", fontsize=12, fontweight="bold")
plt.tight_layout(); plt.savefig(RESULTS / RUN / "physics_maps.png", dpi=130, bbox_inches="tight"); plt.show()

## λ ablation

Train the same architecture with λ ∈ {0, 0.01, 0.1, 1}. λ = 0 is the physics-free baseline.
Compare test macro AUC and the final Poisson residual. Full runs are best done from the CLI:

```bash
for L in 0 0.01 0.1 1; do python scripts/train.py --model pinn --lambda-poisson $L; done
```
This cell summarises whichever runs exist in `results/`.

In [ ]:
import json
rows = []
for d in sorted(RESULTS.glob("pinn_lambda*")):
    if not (d / "metrics.json").exists() or d.name.endswith("_smoke") != SMOKE:
        continue
    m = json.load(open(d / "metrics.json")); h = json.load(open(d / "history.json"))
    lam = m["config"]["extra"].get("lambda_poisson")
    rows.append((lam, m["test"]["macro_auc"], m["test"]["accuracy"], h["val_poisson"][m["best_epoch"] - 1]))
print(f"{'lambda':>8} {'test AUC':>9} {'test acc':>9} {'val Poisson':>12}")
for lam, a, acc, p in sorted(rows):
    print(f"{lam:>8g} {a:>9.4f} {acc:>9.4f} {p:>12.4g}")